# 03 - Embeddings + pgvector

Generate Titan Embeddings V2 (1024 dimensions) for graph nodes stored in Aurora PostgreSQL + Apache AGE,
then persist vectors into the `node_embeddings` pgvector table for similarity search.
All Aurora access uses the RDS Data API — no direct TCP connections required.

## Step 1: Setup — clients, helpers, and embedding function

In [ ]:
import boto3
import json
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION          = config["REGION"]
CLUSTER_ARN     = config["AURORA_CLUSTER_ARN"]
SECRET_ARN      = config["AURORA_SECRET_ARN"]
DB_NAME         = config.get("DB_NAME", "orgiq")
GRAPH_NAME      = config.get("GRAPH_NAME", "orgiq_graph")
EMBEDDING_MODEL = config.get("EMBEDDING_MODEL_ID", "amazon.titan-embed-text-v2:0")
EMBEDDING_DIM   = int(config.get("EMBEDDING_DIMENSIONS", 1024))

if not CLUSTER_ARN:
    raise RuntimeError("AURORA_CLUSTER_ARN not set. Run 02-aurora-graph.ipynb first.")

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
rds_data        = boto3.client("rds-data",        region_name=REGION)
tracker         = ResourceTracker()


# ---------------------------------------------------------------------------
# Helper: execute a SQL statement via the RDS Data API
# ---------------------------------------------------------------------------
def _exec(sql: str, params: list = None) -> dict:
    """Run a SQL statement against Aurora via the RDS Data API."""
    kwargs = dict(
        resourceArn=CLUSTER_ARN,
        secretArn=SECRET_ARN,
        database=DB_NAME,
        sql=sql,
    )
    if params:
        kwargs["parameters"] = params
    return rds_data.execute_statement(**kwargs)


# ---------------------------------------------------------------------------
# Helper: run an openCypher query via Apache AGE
# ---------------------------------------------------------------------------
def _cypher(query: str) -> list[dict]:
    """Execute an openCypher query inside AGE and return rows as dicts."""
    sql = (
        f"SELECT * FROM ag_catalog.cypher('{GRAPH_NAME}', "
        f"$$ {query} $$) as (result ag_catalog.agtype)"
    )
    response = _exec(sql)
    rows = []
    for record in response.get("records", []):
        # Each record is a list with one agtype column; parse the JSON-like value
        raw = record[0].get("stringValue", "{}")
        try:
            rows.append(json.loads(raw))
        except json.JSONDecodeError:
            rows.append({"raw": raw})
    return rows


# ---------------------------------------------------------------------------
# Helper: generate a 1024-dim embedding via Titan Embeddings V2
# ---------------------------------------------------------------------------
def get_embedding(text: str) -> list[float]:
    """Call Titan Embeddings V2 and return a 1024-dimensional float list."""
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL,
        body=json.dumps({
            "inputText": text,
            "dimensions": EMBEDDING_DIM,
            "normalize": True,
        }),
    )
    return json.loads(response["body"].read())["embedding"]


# Smoke test
_test = get_embedding("test")
print(f"\u2713 Config loaded  (cluster={CLUSTER_ARN[-20:]}...)")
print(f"\u2713 Embedding ready (model={EMBEDDING_MODEL}, dim={len(_test)})")

## Step 2: Ensure embedding table

Enable the pgvector extension and create the `node_embeddings` table if it does not already exist.

In [ ]:
# Enable pgvector (idempotent)
_exec("CREATE EXTENSION IF NOT EXISTS vector")
print("\u2713 pgvector extension enabled")

# Create the embeddings table (idempotent)
_exec("""
CREATE TABLE IF NOT EXISTS node_embeddings (
    node_id   TEXT PRIMARY KEY,
    node_type TEXT NOT NULL,
    content   TEXT NOT NULL,
    embedding vector(1024)
)
""")
print("\u2713 node_embeddings table ready")

# Optional: index for fast ANN search (IVFFlat; adjust lists when row count grows)
_exec("""
CREATE INDEX IF NOT EXISTS node_embeddings_ivfflat_idx
    ON node_embeddings
    USING ivfflat (embedding vector_l2_ops)
    WITH (lists = 10)
""")
print("\u2713 IVFFlat index ready")

## Step 3: Embed Person nodes

Query every `Person` node from the AGE graph, build a rich profile string, generate an embedding,
and upsert the result into `node_embeddings`.

In [ ]:
cypher_people = """
MATCH (p:Person)
RETURN p.id as id, p.name as name, p.role as role, p.team as team
"""
people = _cypher(cypher_people)
print(f"\u231b Embedding {len(people)} Person nodes...")

for person in people:
    node_id   = str(person.get("id", ""))
    name      = person.get("name", "Unknown")
    role      = person.get("role", "")
    team      = person.get("team", "")

    profile_text = f"{name}, {role}. Team: {team}."
    embedding    = get_embedding(profile_text)
    vec_str      = "[" + ",".join(f"{v:.8f}" for v in embedding) + "]"

    _exec(
        """
        INSERT INTO node_embeddings (node_id, node_type, content, embedding)
        VALUES (:node_id, 'Person', :content, CAST(:embedding AS vector))
        ON CONFLICT (node_id) DO UPDATE
            SET node_type = EXCLUDED.node_type,
                content   = EXCLUDED.content,
                embedding = EXCLUDED.embedding
        """,
        params=[
            {"name": "node_id",   "value": {"stringValue": node_id}},
            {"name": "content",   "value": {"stringValue": profile_text}},
            {"name": "embedding", "value": {"stringValue": vec_str}},
        ],
    )
    print(f"  \u2713 Embedded Person: {name} ({node_id})")

print(f"\u2713 All Person embeddings stored ({len(people)} rows upserted)")

## Step 4: Embed Skill nodes

Query every `Skill` node, embed name + domain, upsert into `node_embeddings`.

In [ ]:
cypher_skills = """
MATCH (s:Skill)
RETURN s.id as id, s.name as name, s.domain as domain
"""
skills = _cypher(cypher_skills)
print(f"\u231b Embedding {len(skills)} Skill nodes...")

for skill in skills:
    node_id = str(skill.get("id", ""))
    name    = skill.get("name", "Unknown")
    domain  = skill.get("domain", "")

    skill_text = f"{name}" + (f" ({domain})" if domain else "")
    embedding  = get_embedding(skill_text)
    vec_str    = "[" + ",".join(f"{v:.8f}" for v in embedding) + "]"

    _exec(
        """
        INSERT INTO node_embeddings (node_id, node_type, content, embedding)
        VALUES (:node_id, 'Skill', :content, CAST(:embedding AS vector))
        ON CONFLICT (node_id) DO UPDATE
            SET node_type = EXCLUDED.node_type,
                content   = EXCLUDED.content,
                embedding = EXCLUDED.embedding
        """,
        params=[
            {"name": "node_id",   "value": {"stringValue": node_id}},
            {"name": "content",   "value": {"stringValue": skill_text}},
            {"name": "embedding", "value": {"stringValue": vec_str}},
        ],
    )
    print(f"  \u2713 Embedded Skill: {name} ({node_id})")

print(f"\u2713 All Skill embeddings stored ({len(skills)} rows upserted)")

## Step 5: Vector similarity search — People

Use the pgvector `<->` (L2 distance) operator to find people most similar to a free-text query.

In [ ]:
search_text       = "Who has deep expertise in container orchestration and deployment?"
query_embedding   = get_embedding(search_text)
query_vec_str     = "[" + ",".join(f"{v:.8f}" for v in query_embedding) + "]"

response = _exec(
    """
    SELECT node_id,
           content,
           embedding <-> CAST(:query_embedding AS vector) AS distance
    FROM   node_embeddings
    WHERE  node_type = 'Person'
    ORDER  BY distance
    LIMIT  5
    """,
    params=[
        {"name": "query_embedding", "value": {"stringValue": query_vec_str}},
    ],
)

print(f"Query: '{search_text}'\n")
print("Top Person matches:")
for record in response.get("records", []):
    node_id  = record[0].get("stringValue", "?")
    content  = record[1].get("stringValue", "?")
    distance = record[2].get("doubleValue", float("nan"))
    print(f"  [{node_id}]  dist={distance:.4f}  {content}")

print("\n\u2713 Person vector search working")

## Step 6: Vector similarity search — Skills

Same pgvector query but filtered to `node_type = 'Skill'`.

In [ ]:
skill_search_text  = "automated deployment pipelines and CI/CD tooling"
skill_embedding    = get_embedding(skill_search_text)
skill_vec_str      = "[" + ",".join(f"{v:.8f}" for v in skill_embedding) + "]"

response = _exec(
    """
    SELECT node_id,
           content,
           embedding <-> CAST(:query_embedding AS vector) AS distance
    FROM   node_embeddings
    WHERE  node_type = 'Skill'
    ORDER  BY distance
    LIMIT  5
    """,
    params=[
        {"name": "query_embedding", "value": {"stringValue": skill_vec_str}},
    ],
)

print(f"Query: '{skill_search_text}'\n")
print("Top Skill matches:")
for record in response.get("records", []):
    node_id  = record[0].get("stringValue", "?")
    content  = record[1].get("stringValue", "?")
    distance = record[2].get("doubleValue", float("nan"))
    print(f"  [{node_id}]  dist={distance:.4f}  {content}")

print("\n\u2713 Skill vector search working")

## Step 7: Hybrid — vector search + graph traversal

Find the nearest Person vectors via pgvector, then for each result follow `HAS_SKILL`
relationships in the AGE graph to surface their skills.

In [ ]:
hybrid_text    = "security and identity management"
hybrid_emb     = get_embedding(hybrid_text)
hybrid_vec_str = "[" + ",".join(f"{v:.8f}" for v in hybrid_emb) + "]"

# Step A: nearest Person neighbours from pgvector
vec_response = _exec(
    """
    SELECT node_id,
           content,
           embedding <-> CAST(:query_embedding AS vector) AS distance
    FROM   node_embeddings
    WHERE  node_type = 'Person'
    ORDER  BY distance
    LIMIT  3
    """,
    params=[
        {"name": "query_embedding", "value": {"stringValue": hybrid_vec_str}},
    ],
)

candidates = [
    {
        "node_id":  r[0].get("stringValue", ""),
        "content":  r[1].get("stringValue", ""),
        "distance": r[2].get("doubleValue", float("nan")),
    }
    for r in vec_response.get("records", [])
]

print(f"Query: '{hybrid_text}'\n")
print("Hybrid results (vector match + graph skills):")

# Step B: for each candidate, follow HAS_SKILL edges in AGE
for c in candidates:
    pid = c["node_id"]
    skill_rows = _cypher(
        f"MATCH (p:Person {{id: '{pid}'}})-[:HAS_SKILL]->(s:Skill) RETURN s.name as skill_name"
    )
    skills_found = [r.get("skill_name", "") for r in skill_rows]
    skills_str   = ", ".join(skills_found) if skills_found else "(no skills in graph)"
    print(f"  [{pid}]  dist={c['distance']:.4f}  {c['content']}")
    print(f"    Skills: {skills_str}")

print("\n\u2713 Hybrid vector + graph query working")

## Step 8: Coverage check

Count rows in `node_embeddings` grouped by `node_type` to verify all node types were embedded.

In [ ]:
coverage = _exec(
    """
    SELECT node_type, COUNT(*) AS cnt
    FROM   node_embeddings
    GROUP  BY node_type
    ORDER  BY node_type
    """
)

print("Embedding coverage:")
total = 0
for record in coverage.get("records", []):
    node_type = record[0].get("stringValue", "?")
    cnt       = record[1].get("longValue", 0)
    total    += cnt
    print(f"  {node_type}: {cnt} rows")
print(f"  ---------")
print(f"  TOTAL:  {total} rows")
print("\n\u2713 Coverage check complete")

## Step 9: Save to config

In [ ]:
config["EMBEDDINGS_TABLE"] = "node_embeddings"
Path("config.json").write_text(json.dumps(config, indent=2))
print("\u2713 config.json updated:")
print(f"  EMBEDDINGS_TABLE = node_embeddings")

## Summary

- [x] Titan Embeddings V2 generating 1024-dimension vectors via `bedrock-runtime`
- [x] Aurora PostgreSQL accessed exclusively through the RDS Data API (`rds-data` client)
- [x] `node_embeddings` pgvector table created with IVFFlat index
- [x] Person nodes embedded with name + role + team profile text
- [x] Skill nodes embedded with name + domain text
- [x] Vector similarity search working for People (`<->` L2 distance)
- [x] Vector similarity search working for Skills
- [x] Hybrid query: pgvector nearest-neighbour + AGE `HAS_SKILL` graph traversal
- [x] `EMBEDDINGS_TABLE` written to `config.json`

**Next:** [04-knowledge-base.ipynb](04-knowledge-base.ipynb) — Create the Bedrock Knowledge Base backed by Aurora pgvector for RAG